In [19]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import numpy as np
import os
import networkx as nx
import folium
from datetime import datetime
from dataclasses import dataclass, field
from enum import Enum, auto
import heapq
import random
import csv
from collections import deque
from typing import List, Optional, Any, Dict
import traceback


#  1. GRAFO Y MAPA  - OPTIMIZADO CON CACHE


class CityGraph:
    """
    Gestiona la representación de la red urbana mediante un grafo dirigido.
    
    Implementa el patrón de diseño Singleton para asegurar que la matriz espacial 
    se instancie una única vez en memoria, optimizando el rendimiento global del sistema. 
    Incorpora un mecanismo de caché heurístico para almacenar las rutas mínimas 
    calculadas previamente (mediante el algoritmo de Dijkstra), lo que reduce 
    significativamente el costo computacional de las consultas reiterativas 
    durante la ejecución de las múltiples réplicas del experimento.
    """
    _instance = None
    def __new__(cls):
        if cls._instance is None: 
            cls._instance = super(CityGraph, cls).__new__(cls)
            cls._instance._initialized = False
        return cls._instance

    def __init__(self):
        if self._initialized: return
        self.coords = {}
        self.df_nodos = None
        self.G = None
        self.status = "Iniciando..."
        self._rutas_cache = {}  #  Cache de rutas
        self._cargar_datos()
        self._initialized = True
    
    def _cargar_datos(self):
        try:
            if not os.path.exists("nodos.csv"): 
                self.status = "❌ Falta 'nodos.csv'"
                return
            
            with open("nodos.csv", "r") as f:
                line = f.readline()
                sep = ";" if ";" in line else ","
            
            df = pd.read_csv("nodos.csv", sep=sep)
            
            # Normalizar columnas
            cols = {c: c.lower() for c in df.columns}
            rename = {}
            for r, l in cols.items(): 
                if 'id' in l: rename[r] = 'Nodo_ID'
                if 'lat' in l: rename[r] = 'Latitud'
                if 'lon' in l: rename[r] = 'Longitud'
            df.rename(columns=rename, inplace=True)
            
            for c in ['Latitud', 'Longitud']: 
                if df[c].dtype == object: 
                    df[c] = df[c].str.replace(',', '.', regex=False).astype(float)

            df = df.drop_duplicates(subset=['Nodo_ID'], keep='first') 
            self.coords = df.set_index('Nodo_ID')[['Latitud', 'Longitud']].to_dict('index')
            
            if os.path.exists("arcos_con_distancia.csv"):
                df_e = pd.read_csv("arcos_con_distancia.csv")
                self.G = nx.from_pandas_edgelist(
                    df_e, source='Nodo_Origen', target='Nodo_Destino', 
                    edge_attr='Peso', create_using=nx.DiGraph()
                )
            else: 
                self.G = nx.DiGraph()
            
            self.status = f"✅ Datos OK: {len(self.coords)} nodos."
        except Exception as e: 
            self.status = f"❌ Error: {e}"

    def get_rutas_desde(self, origen, cutoff):
        """Calcula distancias  usando Dijkstra """
        # 1. Verificar Cache
        cache_key = (origen, cutoff)
        if cache_key in self._rutas_cache:
            return self._rutas_cache[cache_key]

        # 2. Calcular si no está en cache
        if not self.G: return {}, {}, []
        try:
            dists, rutas = nx.single_source_dijkstra(self.G, source=origen, weight='Peso', cutoff=cutoff)
            validos = list(dists.keys())
            if origen in validos: validos.remove(origen)
            
            # 3. Guardar y retornar
            resultado = (dists, rutas, validos)
            self._rutas_cache[cache_key] = resultado
            return resultado
        except: 
            return {}, {}, []

    def get_lat_lon(self, nid):
        if nid in self.coords: 
            return self.coords[nid]['Latitud'], self.coords[nid]['Longitud']
        return None, None


#  2. CLASES DE CONFIGURACIÓN Y DATOS


class DistType(Enum): 
    EXPONENCIAL="Exp"
    NORMAL="Norm"
    UNIFORME="Unif"
    CONSTANTE="Fijo"

class EventType(Enum): 
    LLEGADA_GLOBAL=auto()
    LLEGADA_LOCAL=auto()
    FIN_COCINA=auto()
    VUELTA_MOTOQUERO=auto()

@dataclass(order=True)
class Event: 
    time: float
    priority: int
    event_type: EventType = field(compare=False)
    data: Dict[str, Any] = field(compare=False, default_factory=dict)

@dataclass
class GlobalConfig: 
    dias_a_simular: int
    horas_turno: float
    horas_cierre_pedidos: float
    velocidad_moto_kmh: float
    replicas: int
    modo_centralizado: bool
    map_limit: int
    seed: int
    replica_start_id: int 

@dataclass
class RestaurantConfig: 
    nodo_id: int
    radio_km: float
    num_cocineros: int
    num_motoqueros: int
    dist_llegada_tipo: DistType; dist_llegada_p1: float; dist_llegada_p2: float
    dist_cocina_tipo: DistType; dist_cocina_p1: float; dist_cocina_p2: float
    dist_descanso_tipo: DistType; dist_descanso_p1: float; dist_descanso_p2: float

class DistributionManager:
    """
    Gestor estocástico del modelo de simulación de eventos discretos.
    
    Responsable de la generación de variables pseudoaleatorias para modelar 
    la variabilidad de los procesos logísticos, tales como tasas de llegada, 
    tiempos de preparación (WIP) y periodos de recuperación de flota. Utiliza 
    semillas independientes (función hash) inyectadas desde los controladores, 
    garantizando así la total independencia estocástica y la reproducibilidad 
    estadística transversal entre las distintas corridas y escenarios evaluados.
    """
    @staticmethod
    def generar(rng, tipo, p1, p2=0.0):
        if tipo == DistType.EXPONENCIAL: return rng.expovariate(1/p1) if p1>0 else 999
        if tipo == DistType.NORMAL: return rng.normalvariate(p1, p2)
        if tipo == DistType.UNIFORME: return rng.uniform(p1, p2)
        if tipo == DistType.CONSTANTE: return p1
        return p1

@dataclass
class Pedido:
    """
    Objeto de información del modelo de simulación.
    
    Actúa como el vector principal de trazabilidad de datos. Almacena de manera dinámica 
    los atributos espaciales (coordenadas topológicas de origen y destino) y registra 
    las marcas temporales precisas (timestamps) en cada transición de estado del ciclo de servicio. 
    Esta recolección  permite la reconstrucción y el cálculo de los 
    Indicadores Clave de Desempeño (KPIs) logísticos, fundamentalmente el Lead Time total 
    del sistema, los tiempos de espera y el nivel de ocupación de 
    los recursos .
    """
    id: int
    replica_id: int
    dia_simulacion: int
    restaurante_origen: int
    nodo_destino: int
    distancia_km: float
    ruta_nodos: List[int]
    t_llegada: float
    
    # KPIs
    cola_cocina_len: int = 0
    cola_despacho_len: int = 0
    sede_mas_cercana_id: int = 0
    motos_libres_al_llegar: int=0
    cocineros_libres_al_llegar: int=0
    
    # Timestamps
    t_inicio_cocina: float=0
    t_fin_cocina: float=0
    t_inicio_despacho: float=0
    t_entrega_cliente: float=0
    t_regreso_moto: float=0
    t_moto_libre: float=0
    
    def espera_cocina(self): return max(0, self.t_inicio_cocina - self.t_llegada)
    def tiempo_preparacion(self): return max(0, self.t_fin_cocina - self.t_inicio_cocina)
    def tiempo_ciclo_total(self): return max(0, self.t_entrega_cliente - self.t_llegada) if self.t_entrega_cliente else 0
    def espera_moto_listo(self): return max(0, self.t_inicio_despacho - self.t_fin_cocina) if self.t_fin_cocina else 0

class Recurso:
    def __init__(self, id_): 
        self.id = id_
        self.ocupado = False
        self.t_liberacion = 0.0
    
    def liberar(self): 
        self.ocupado = False
    
    def ocupar_hasta(self, t_fin): 
        self.ocupado = True
        self.t_liberacion = t_fin

class Motoquero(Recurso):
    def __init__(self, id_, vel): 
        super().__init__(id_)
        self.velocidad = vel


#  3. MODELO DE SIMULACIÓ -


class Restaurant:
    """
    Entidd que modela el comportamiento operativo e infraestructura de una sede.
    
    A nivel de micro-simulación, esta clase administra las estructuras de cola operacinales 
    (bajo diciplina de atención estricta FIFO) para las sub-áreas de preparación
    y de despacho. Es responsable de gestionar la restricción de capacidad finita 
    de los servidores del sistema (pool de operarios de cocina y dotación de flota de 
    repartidores), capturando y programando dinámicamente los eventos vinculados 
    a las tasas de liberación de los procesos.
    """
    def __init__(self, cfg, g_cfg, sim):
        self.cfg = cfg
        self.g_cfg = g_cfg
        self.sim = sim
        self.nodo_id = cfg.nodo_id
        
        # Carga del Grafo Local
        self.cg = CityGraph()
        self.dists, self.rutas, self.nodos_validos = self.cg.get_rutas_desde(self.nodo_id, self.cfg.radio_km)
        
        # Validación de Sede
        if not self.nodos_validos:
            raise ValueError(
                f"\n❌ ERROR FATAL: Sede {self.nodo_id} NO tiene clientes cerca.\n"
                f"   Radio: {self.cfg.radio_km} km | Nodos encontrados: 0\n"
                f"   SOLUCIÓN: Aumenta el radio o cambia el nodo sede."
            )

        #  Semilla Hash Real
        self.rng = random.Random(hash((self.g_cfg.seed, sim.current_rep_id, self.nodo_id, "rest")))
        
        self.reiniciar_dia()
    
    def reiniciar_dia(self):
        self.cola_cocina = deque()
        self.cola_despacho = deque()
        self.cocineros = [Recurso(i) for i in range(self.cfg.num_cocineros)]
        self.motoqueros = [Motoquero(i, self.g_cfg.velocidad_moto_kmh) for i in range(self.cfg.num_motoqueros)]
        self.pedidos_dia_actual = []
        self.cnt_id = 0
        self.cierre_real = 0.0 
    
    def generar_pedido_local(self, t, rep, dia):
        if not self.nodos_validos: return
        self.cnt_id += 1
        dest = self.rng.choice(self.nodos_validos)
        
        p = Pedido(self.cnt_id, rep, dia, self.nodo_id, dest, self.dists.get(dest,1), self.rutas.get(dest,[]), t)
        
        # Guardar Estado del Sistema
        p.cocineros_libres_al_llegar = sum(1 for c in self.cocineros if not c.ocupado)
        p.motos_libres_al_llegar = sum(1 for m in self.motoqueros if not m.ocupado)
        p.cola_cocina_len = len(self.cola_cocina)
        p.cola_despacho_len = len(self.cola_despacho)
        p.sede_mas_cercana_id = self.nodo_id 
        
        self.cola_cocina.append(p)
        self.asignar_cocina()

    def asignar_cocina(self):
        chef = next((c for c in self.cocineros if not c.ocupado), None)
        if chef and self.cola_cocina:
            ped = self.cola_cocina.popleft()
            dur = DistributionManager.generar(self.rng, self.cfg.dist_cocina_tipo, self.cfg.dist_cocina_p1, self.cfg.dist_cocina_p2)
            
            t_fin = self.sim.t_actual + dur
            chef.ocupar_hasta(t_fin)
            ped.t_inicio_cocina = self.sim.t_actual
            
            self.sim.add_event(t_fin, EventType.FIN_COCINA, {'rest':self.nodo_id, 'chef':chef.id, 'ped':ped})
    
    def fin_cocina(self, chef_id, ped):
        self.cocineros[chef_id].liberar()
        ped.t_fin_cocina = self.sim.t_actual
        self.cola_despacho.append(ped)
        self.asignar_cocina()
        self.asignar_moto()
    
    def asignar_moto(self):
        moto = next((m for m in self.motoqueros if not m.ocupado), None)
        if moto and self.cola_despacho:
            ped = self.cola_despacho.popleft()
            
            viaje = (ped.distancia_km / moto.velocidad) * 60
            ped.t_inicio_despacho = self.sim.t_actual
            ped.t_entrega_cliente = self.sim.t_actual + viaje
            t_regreso = ped.t_entrega_cliente + viaje
            ped.t_regreso_moto = t_regreso
            
            desc = DistributionManager.generar(self.rng, self.cfg.dist_descanso_tipo, self.cfg.dist_descanso_p1, self.cfg.dist_descanso_p2)
            ped.t_moto_libre = t_regreso + desc
            
            moto.ocupar_hasta(ped.t_moto_libre) 
            
            if ped.t_moto_libre > self.cierre_real: 
                self.cierre_real = ped.t_moto_libre 
            
            self.sim.add_event(ped.t_moto_libre, EventType.VUELTA_MOTOQUERO, {'rest':self.nodo_id, 'moto':moto.id, 'ped':ped})
    
    def vuelta_moto(self, mid, ped):
        self.motoqueros[mid].liberar()
        self.pedidos_dia_actual.append(ped)
        self.asignar_moto()

class CentralDispatcher:
    """
    Motor heurístico de ruteo y asignación centralizada.
    
    A diferencia de la asignación estática tradicional, este algoritmo incorpora una 
    regla de decisión dinámica que proyecta la integralidad del instante de servicio 
    futuro para toda la red. Evalúa de forma simultánea e iterativa el Work In Process (WIP) 
    actual de cada bloque de preparación, proyecta el perfil de disponibilidad estimada de 
    la flota de reparto y cruza dicha carga de datos con la matriz de distancias 
    topológicas reales (km). Finalmente, enruta y asigna cada orden de demanda aleatoria 
    a la sede que garantice matemáticamente el menor Lead Time total de entrega.
    """
    def __init__(self, sim, rests):
        self.sim = sim
        self.rests = rests
        self.pool = list(set().union(*[r.nodos_validos for r in rests]))
        # Semilla Hash
        self.rng = random.Random(hash((sim.g.seed, sim.current_rep_id, "central")))
    
    def generar_pedido(self, t, rep, dia):
        if not self.pool: return
        dest = self.rng.choice(self.pool)
        
        mejor = None
        score_min = float('inf')
        
        dist_min_real = float('inf')
        id_mas_cercano = -1

        for r in self.rests:
            if dest not in r.nodos_validos: continue
            
            dist_km = r.dists.get(dest, 99)
            if dist_km < dist_min_real: 
                dist_min_real = dist_km
                id_mas_cercano = r.nodo_id

            # --- SCORE (CON DESCANSO) ---
            trabajo_en_hornos = sum(max(0, c.t_liberacion - t) for c in r.cocineros)
            trabajo_en_cola = len(r.cola_cocina) * r.cfg.dist_cocina_p1
            carga_total = trabajo_en_hornos + trabajo_en_cola
            
            if carga_total < 1e-5:
                t_inicio_cocina = t
            else:
                t_inicio_cocina = t + (carga_total / max(1, r.cfg.num_cocineros))
            
            t_fin_cocina_est = t_inicio_cocina + r.cfg.dist_cocina_p1

            #  Estimación con Descanso Promedio
            motos_libres = [m for m in r.motoqueros if not m.ocupado]
            
            if motos_libres:
                t_inicio_moto = t_fin_cocina_est
            else:
                tiempos_lib = [m.t_liberacion for m in r.motoqueros if m.ocupado]
                if not tiempos_lib:
                    t_inicio_moto = t_fin_cocina_est + 9999
                else:
                    prox_lib = min(tiempos_lib)
                    # Sumar descanso esperado según config
                    descanso_est = 0
                    if r.cfg.dist_descanso_tipo == DistType.UNIFORME:
                        descanso_est = (r.cfg.dist_descanso_p1 + r.cfg.dist_descanso_p2) / 2.0
                    elif r.cfg.dist_descanso_tipo == DistType.NORMAL:
                        descanso_est = r.cfg.dist_descanso_p1
                    elif r.cfg.dist_descanso_tipo == DistType.EXPONENCIAL:
                        descanso_est = r.cfg.dist_descanso_p1
                    else:
                        descanso_est = r.cfg.dist_descanso_p1
                    
                    t_inicio_moto = max(t_fin_cocina_est, prox_lib + descanso_est)

            t_viaje = (dist_km * 60) / r.g_cfg.velocidad_moto_kmh
            score = t_inicio_moto + t_viaje
            
            if score < score_min: 
                score_min = score
                mejor = r
            
        if mejor:
            mejor.cnt_id += 1
            p = Pedido(mejor.cnt_id, rep, dia, mejor.nodo_id, dest, mejor.dists.get(dest,1), mejor.rutas.get(dest,[]), t)
            
            p.cocineros_libres_al_llegar = sum(1 for c in mejor.cocineros if not c.ocupado)
            p.motos_libres_al_llegar = sum(1 for m in mejor.motoqueros if not m.ocupado)
            p.cola_cocina_len = len(mejor.cola_cocina)
            p.cola_despacho_len = len(mejor.cola_despacho)
            p.sede_mas_cercana_id = id_mas_cercano 

            mejor.cola_cocina.append(p)
            mejor.asignar_cocina()

class Simulator:
    """
    Arquitectura central del framework de Simulación de Eventos Discretos (DES).
    
    Esta clase actúa como el orquestador temporal del sistema estocástico. Se encarga de 
    administrar el reloj de simulación de manera continua y coordina de forma dinámica la 
    Lista de Eventos Futuros (FEL), implementada a bajo nivel sobre una estructura de datos 
    de tipo Heap Binario para asegurar una extracción y procesamiento en orden cronológico O(log n). 
    Adicionalmente, gestiona de forma nativa la exportación masiva de la bitácora de datos (Logs) 
    implementando una rutina  por lotes (batch processing), 
    mitigando el riesgo de  saturación en la asignación de memoria RAM 
    ante experimentos que demandan altos volúmenes de iteración.
    """
    def __init__(self, g_cfg, r_cfgs):
        self.g = g_cfg
        self.r_cfgs = r_cfgs
        self.t_actual = 0
        self.eventos = []
        self.last_run_restaurants = {}
        self.cg = CityGraph() 
        
        self.timestamp = datetime.now().strftime('%Y%m%d_%H%M')
        self.part_count = 1
        self.rows_in_current_part = 0
        self.max_rows_per_file = 1000000 
        self.current_rep_id = 0 
        
        self.current_pedidos_file = f"Tesis_Detalle_{self.timestamp}_Parte1.csv"
        self.resumen_file = f"Tesis_Resumen_{self.timestamp}.csv"
        self._init_files()

    def _init_files(self):
        with open(self.resumen_file, 'w', newline='', buffering=1024*1024) as f:
            writer = csv.writer(f)
            writer.writerow(["Replica", "Dia", "Sede", "Total_Pedidos", "Overtime_Min", "Dias_Sim_Total", "Replicas_Total", "Horas_Turno", "Centralizado", "Config_Cocineros", "Config_Motos"])
        self._create_new_pedidos_file()

    def _create_new_pedidos_file(self):
        with open(self.current_pedidos_file, 'w', newline='', buffering=1024*1024) as f:
            writer = csv.writer(f)
            writer.writerow(["Replica", "Dia", "Sede_Origen", "ID_Pedido", "Nodo_Destino", "Distancia_KM", 
                             "Motos_Libres_Al_Llegar", "Cocineros_Libres_Al_Llegar",
                             "Cola_Cocina_En_Llegada", "Cola_Despacho_En_Llegada", "Sede_Mas_Cercana_ID",
                             "T_Llegada", "T_Inicio_Cocina", "T_Fin_Cocina", "T_Salida_Reparto", 
                             "T_Entrega_Cliente", "T_Regreso_Sede", "T_Moto_Liberada",
                             "Espera_Cocina_Min", "T_Preparacion_Min", "Espera_Moto_Lista_Min", 
                             "T_Viaje_Ida_Min", "T_Ciclo_Total_Min"])

    def add_event(self, t, tipo, data): 
        heapq.heappush(self.eventos, Event(t, 0, tipo, data))
    
    def run_full(self):
        start = self.g.replica_start_id
        end = start + self.g.replicas
        tot = self.g.replicas * self.g.dias_a_simular
        pb = widgets.IntProgress(min=0, max=tot, description=f'Simulando Rep {start} a {end-1}:')
        display(pb)
        
        for rep in range(start, end):
            self.current_rep_id = rep
            try:
                rests = {rc.nodo_id: Restaurant(rc, self.g, self) for rc in self.r_cfgs}
                
                for dia in range(1, self.g.dias_a_simular+1):
                    self.run_day(rep, dia, rests)
                    self._flush_data(rep, dia, rests)
                    pb.value += 1
                
                self.last_run_restaurants = rests 
            
            except Exception as e:
                print(f"⚠️ ERROR FATAL en Réplica {rep}: {e}")
                traceback.print_exc()
                continue 
            
        return f"✅ Lote Completo . Resumen: {self.resumen_file}."

    def run_day(self, rep, dia, rests):
        self.t_actual = 0
        self.eventos = []
        disp = CentralDispatcher(self, list(rests.values())) if self.g.modo_centralizado else None
        
        if self.g.modo_centralizado:
            lam = sum(1/r.cfg.dist_llegada_p1 for r in rests.values() if r.cfg.dist_llegada_p1 > 0)
            if lam > 0: 
                self.add_event(random.expovariate(lam), EventType.LLEGADA_GLOBAL, {'lam':lam})
        else:
            for r in rests.values(): 
                self.add_event(DistributionManager.generar(r.rng, r.cfg.dist_llegada_tipo, r.cfg.dist_llegada_p1), EventType.LLEGADA_LOCAL, {'rest':r.nodo_id})
        
        while self.eventos:
            ev = heapq.heappop(self.eventos)
            self.t_actual = ev.time
            limit = self.g.horas_cierre_pedidos * 60
            
            if ev.event_type == EventType.LLEGADA_GLOBAL:
                if self.t_actual < limit:
                    disp.generar_pedido(self.t_actual, rep, dia)
                    self.add_event(self.t_actual + random.expovariate(ev.data['lam']), EventType.LLEGADA_GLOBAL, ev.data)
            
            elif ev.event_type == EventType.LLEGADA_LOCAL:
                r = rests.get(ev.data['rest'])
                if r and self.t_actual < limit:
                    r.generar_pedido_local(self.t_actual, rep, dia)
                    dt = DistributionManager.generar(r.rng, r.cfg.dist_llegada_tipo, r.cfg.dist_llegada_p1)
                    self.add_event(self.t_actual + dt, EventType.LLEGADA_LOCAL, ev.data)
            
            else:
                r = rests.get(ev.data['rest'])
                if r:
                    if ev.event_type == EventType.FIN_COCINA: 
                        r.fin_cocina(ev.data['chef'], ev.data['ped'])
                    elif ev.event_type == EventType.VUELTA_MOTOQUERO: 
                        r.vuelta_moto(ev.data['moto'], ev.data['ped'])

    # Escritura Batch
    def _flush_data(self, rep, dia, rests):
        pedidos_rows = []
        resumen_rows = []
        
        for r in rests.values():
            for p in r.pedidos_dia_actual:
                t_ida = (p.distancia_km / self.g.velocidad_moto_kmh) * 60
                pedidos_rows.append([
                    rep, dia, r.nodo_id, p.id, p.nodo_destino, 
                    round(p.distancia_km, 3),
                    p.motos_libres_al_llegar, p.cocineros_libres_al_llegar,
                    p.cola_cocina_len, p.cola_despacho_len, p.sede_mas_cercana_id,
                    round(p.t_llegada, 2), round(p.t_inicio_cocina, 2), 
                    round(p.t_fin_cocina, 2), round(p.t_inicio_despacho, 2),
                    round(p.t_entrega_cliente, 2), round(p.t_regreso_moto, 2), 
                    round(p.t_moto_libre, 2),
                    round(p.espera_cocina(), 2), round(p.tiempo_preparacion(), 2),
                    round(p.espera_moto_listo(), 2), round(t_ida, 2), 
                    round(p.tiempo_ciclo_total(), 2)
                ])
            
            ovt = max(0, r.cierre_real - (self.g.horas_turno * 60))
            resumen_rows.append([
                rep, dia, r.nodo_id, len(r.pedidos_dia_actual), 
                round(ovt, 2), self.g.dias_a_simular, self.g.replicas,
                self.g.horas_turno, "SI" if self.g.modo_centralizado else "NO",
                r.cfg.num_cocineros, r.cfg.num_motoqueros
            ])

        if self.rows_in_current_part >= self.max_rows_per_file:
            self.part_count += 1
            self.current_pedidos_file = f"Tesis_Detalle_{self.timestamp}_Parte{self.part_count}.csv"
            self._create_new_pedidos_file()
            self.rows_in_current_part = 0

        # Escritura MASIVA
        with open(self.current_pedidos_file, 'a', newline='', buffering=8*1024*1024) as f_ped:
            csv.writer(f_ped).writerows(pedidos_rows)
            self.rows_in_current_part += len(pedidos_rows)
            
        with open(self.resumen_file, 'a', newline='', buffering=8*1024*1024) as f_res:
            csv.writer(f_res).writerows(resumen_rows)
            
        # Limpieza
        if dia < self.g.dias_a_simular:
            for r in rests.values():
                r.pedidos_dia_actual = []

    def get_map(self):
        if not self.last_run_restaurants: return None
        r1 = list(self.last_run_restaurants.values())[0]
        lat, lon = r1.cg.get_lat_lon(r1.nodo_id)
        m = folium.Map([lat, lon], zoom_start=11)
        colors = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'cadetblue']
        idx = 0
        
        for r in self.last_run_restaurants.values():
            col = colors[idx % len(colors)]
            idx += 1
            p = r.cg.get_lat_lon(r.nodo_id)
            
            folium.Marker(p, popup=f"Sede {r.nodo_id}", icon=folium.Icon(color=col, icon='home'), tooltip=f"Sede {r.nodo_id}").add_to(m)
            folium.Circle(p, radius=r.cfg.radio_km*1000, color=col, fill=True, fill_opacity=0.05).add_to(m)
            
            count = 0
            for ped in r.pedidos_dia_actual:
                if count >= self.g.map_limit: break
                if ped.ruta_nodos and len(ped.ruta_nodos) > 1:
                    pts = [r.cg.get_lat_lon(n) for n in ped.ruta_nodos if r.cg.get_lat_lon(n)[0]]
                    if len(pts) > 1: 
                        folium.PolyLine(pts, color=col, weight=2, opacity=0.6).add_to(m)
                        count += 1
        return m


#  4. INTERFAZ DE USUARIO (UI)


cg = CityGraph()
out_mapa_total = widgets.Output()

def mostrar_mapa_completo():
    if not cg.coords: 
        return widgets.HTML("Sin datos")
    
    m = folium.Map(location=[-33.45, -70.65], zoom_start=11)
    print(f"⏳ Dibujando {len(cg.coords)} nodos...")
    
    for nid, d in cg.coords.items(): 
        folium.Circle(
            [d['Latitud'], d['Longitud']], 
            radius=40, 
            color='#3498db', 
            fill=True, 
            fill_opacity=0.6, 
            weight=0, 
            tooltip=f"ID: {nid}"
        ).add_to(m)
    return m

btn_load_map = widgets.Button(
    description="🌍 Cargar Mapa Global (Ver Nodos)", 
    button_style='primary', 
    layout=widgets.Layout(width='98%')
)

def on_load_map(b): 
    out_mapa_total.clear_output()
    with out_mapa_total: 
        display(mostrar_mapa_completo())

btn_load_map.on_click(on_load_map)

# Elementos de Configuración 
style = {'description_width': 'initial'}
ly = widgets.Layout(width='95%')

w_ids = widgets.Text(description="🏠 IDs Sedes (sep ,):", placeholder="Ej: 14834, 14651", style=style, layout=ly)
w_central = widgets.Checkbox(value=True, description="Asignación Centralizada", style=style)

w_dias = widgets.IntText(value=365, description="Días Simulación:", style=style)
w_reps = widgets.IntText(value=50, description="Réplicas por Lote:", style=style)
w_rep_start = widgets.IntText(value=1, description="N° Réplica Inicial:", style=style)
w_map_limit = widgets.IntText(value=50, description="Rutas Mapa:", style=style)
w_vel = widgets.FloatText(value=35.0, description="Vel (km/h):", style=style)

w_turno = widgets.FloatText(value=8.0, description="Turno (hrs):", disabled=False, style=style)
w_cierre = widgets.FloatText(value=7.5, description="Cierre (hrs):", disabled=False, style=style)

btn_cfg = widgets.Button(description="🛠️ Configurar Parámetros", button_style='info', layout=ly)

tabs = widgets.Tab()
out_run = widgets.Output()
widgets_db = {}

def crear_panel_completo(nid):
    head = widgets.HTML(f"<h3>🏠 Configuración Sede {nid}</h3>")
    w_radio = widgets.FloatText(value=5.0, description="Radio (km):", style=style)
    w_cooks = widgets.IntText(value=3, description="Cocineros:", style=style) # <-- Default optimizado a 3
    w_motos = widgets.IntText(value=3, description="Motos:", style=style)
    
    def crear_dist_ui(titulo, def_type, p1, p2):
        lbl = widgets.HTML(f"<b>--- {titulo} ---</b>")
        dt = widgets.Dropdown(options=[(e.value, e) for e in DistType], value=def_type, description="Tipo:", style=style)
        v1 = widgets.FloatText(value=p1, description="Media (min):", style=style)
        v2 = widgets.FloatText(value=p2, description="Desviación:", style=style)
        
        def on_change_type(change): 
            sel = change['new']
            if sel == DistType.UNIFORME:
                v1.description = "Mínimo (min):"
            else:
                v1.description = "Media (min):"
                
            if sel == DistType.EXPONENCIAL or sel == DistType.CONSTANTE:
                v2.layout.display = 'none' 
            else:
                v2.layout.display = 'flex'
        
        dt.observe(on_change_type, names='value')
        on_change_type({'new': def_type})
        return [lbl, dt, v1, v2], {'t':dt, 'p1':v1, 'p2':v2}

    ui_arr, db_arr = crear_dist_ui("Llegada Pedidos", DistType.EXPONENCIAL, 10.0, 0.0)
    ui_kit, db_kit = crear_dist_ui("Tiempo Cocina", DistType.NORMAL, 14.0, 3.0)
    ui_rest, db_rest = crear_dist_ui("Tiempo Descanso", DistType.UNIFORME, 2.0, 5.0)
    
    panel = widgets.VBox([head, w_radio, w_cooks, w_motos] + ui_arr + ui_kit + ui_rest)
    return panel, {'radio': w_radio, 'cooks': w_cooks, 'motos': w_motos, 'arr': db_arr, 'kit': db_kit, 'rest': db_rest}

def on_cfg_click(b):
    global widgets_db
    widgets_db = {}
    with out_run:
        clear_output()
        txt = w_ids.value.strip()
        if not txt: 
            print("⚠️ Escribe IDs de sedes válidos.")
            return
        try:
            ids = [int(x.strip()) for x in txt.split(',') if x.strip().isdigit()]
            children = []
            titles = []
            for nid in ids: 
                p, db = crear_panel_completo(nid)
                children.append(p)
                titles.append(str(nid))
                widgets_db[nid] = db
            
            tabs.children = children
            for i, t in enumerate(titles): 
                tabs.set_title(i, t)
            print("✅ Paneles Genrados. Ajustar las distribuciones abajo.")
        except: 
            print("❌ Error en formato de IDs")

btn_cfg.on_click(on_cfg_click)

btn_run = widgets.Button(description="▶️ EJECUTAR LOTE ", button_style='success', layout=ly)

def on_run_click(b):
    with out_run:
        clear_output()
        if not widgets_db: 
            print("⚠️ Configura sedes primero.")
            return
        
        g_cfg = GlobalConfig(
            w_dias.value, w_turno.value, w_cierre.value, w_vel.value, 
            w_reps.value, w_central.value, w_map_limit.value, 
            seed=42, replica_start_id=w_rep_start.value
        )
        
        r_cfgs = []
        for nid, db in widgets_db.items():
            r_cfgs.append(RestaurantConfig(
                nid, db['radio'].value, db['cooks'].value, db['motos'].value, 
                db['arr']['t'].value, db['arr']['p1'].value, db['arr']['p2'].value, 
                db['kit']['t'].value, db['kit']['p1'].value, db['kit']['p2'].value, 
                db['rest']['t'].value, db['rest']['p1'].value, db['rest']['p2'].value
            ))
            
        sim = Simulator(g_cfg, r_cfgs)
        print(sim.run_full())
        
        m = sim.get_map()
        if m: display(m)

btn_run.on_click(on_run_click)

# Armado Final de la UI
ui = widgets.VBox([
    widgets.HTML("<h2>📍 Simulador Tesis </h2>"),
    btn_load_map, 
    out_mapa_total, 
    widgets.HTML("<hr>"),
    w_ids, 
    btn_cfg, 
    tabs, 
    widgets.HTML("<hr>"), 
    w_central, 
    widgets.HBox([w_dias, w_reps, w_rep_start, w_map_limit]), 
    widgets.HBox([w_turno, w_cierre, w_vel]), 
    btn_run, 
    out_run
])

display(ui)